# Finding the Signal: San Francisco Overdose Deaths (2020–2025)

### A short, structured data story

**Research Question**  
*How do overdose deaths vary by month when we aggregate 2020–2025 together, and what months appear riskier?*

**Dataset used**  
`overdose_deaths_by_month_normalized.csv` — 12 rows (one for each month aggregated across 2020–2025) with:
- `month` — label like *January 2020–2025*
- `Total Deaths` — sum of deaths in that month across 2020–2025
- `Normalized Deaths` — min-max scaling of `Total Deaths` to 0–1

---
#### What you'll find in this notebook
1) **Load & preview the data**  
2) **Exploratory Data Analysis (EDA)** — distributions, trend line, simple stats  
3) **Quick statistical checks** — one-sample t‑test vs. reference mean, simple probability  
4) **Key insights & next steps** — concise takeaways you can paste into your report


In [ ]:
# 1) Set-up
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from google.colab import files
sns.set(style="whitegrid")
plt.rcParams["figure.dpi"] = 120
print("Libraries loaded.")

## 2) Load your data
Upload `overdose_deaths_by_month_normalized.csv` when prompted, **or** edit the path if it's already in your runtime.

In [ ]:
try:
    uploaded = files.upload()  # Use the dialog to upload the csv
    fname = list(uploaded.keys())[0]
except Exception:
    # Fallback (edit this path if the file is already present in your drive/runtime)
    fname = 'overdose_deaths_by_month_normalized.csv'

df = pd.read_csv(fname)
df.head()

### Quick data dictionary
- **month** *(str)* – Month label aggregated across years.
- **Total Deaths** *(int/float)* – Sum of deaths in that month across 2020–2025.
- **Normalized Deaths** *(float)* – Min‑max normalized value of `Total Deaths`.

In [ ]:
# 3) Sanity checks
print("Shape:", df.shape)
print("\nDtypes:\n", df.dtypes)
print("\nNulls per column:\n", df.isnull().sum())

display(df[['Total Deaths','Normalized Deaths']].describe().T)

## 4) EDA — Distributions & Trends

In [ ]:
# Histogram of Total Deaths (12 monthly aggregates)
plt.figure(figsize=(8,5))
sns.histplot(df["Total Deaths"], bins=10, kde=True)
plt.title("Distribution of Monthly Overdose Deaths (2020–2025 Aggregated)")
plt.xlabel("Total Deaths")
plt.ylabel("Frequency")
plt.show()

In [ ]:
# Order months Jan..Dec for a clean seasonal line
order = [
    'January 2020–2025','February 2020–2025','March 2020–2025','April 2020–2025',
    'May 2020–2025','June 2020–2025','July 2020–2025','August 2020–2025',
    'September 2020–2025','October 2020–2025','November 2020–2025','December 2020–2025'
]
df_sorted = df.copy()
df_sorted["month"] = pd.Categorical(df_sorted["month"], categories=order, ordered=True)
df_sorted = df_sorted.sort_values("month")

plt.figure(figsize=(10,5))
sns.lineplot(x="month", y="Total Deaths", data=df_sorted, marker="o")
plt.title("Trend of Monthly Overdose Deaths (Aggregated 2020–2025)")
plt.xlabel("Month")
plt.ylabel("Total Deaths")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Bar chart: highlight highest and lowest months
top3 = df_sorted.nlargest(3, 'Total Deaths')
bot3 = df_sorted.nsmallest(3, 'Total Deaths')
print("Top 3 months by total deaths:\n", top3[['month','Total Deaths']])
print("\nBottom 3 months by total deaths:\n", bot3[['month','Total Deaths']])

plt.figure(figsize=(9,5))
sns.barplot(x='month', y='Total Deaths', data=df_sorted)
plt.title('Monthly Overdose Deaths (2020–2025 Aggregated)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 5) Quick statistical checks
These are not meant to be exhaustive—just quick, defensible checks that complement the EDA.

In [ ]:
# One-sample t-test: Is the mean month >= a reference baseline?
# Choose a simple baseline like 320 deaths (roughly the sample mean in many runs).
reference_mean = 320
t_stat, p_val = stats.ttest_1samp(df["Total Deaths"], popmean=reference_mean)
print(f"T-statistic: {t_stat:.3f}, P-value: {p_val:.3f}")
if p_val < 0.05:
    print(f"Conclusion: average monthly deaths differ significantly from {reference_mean}.")
else:
    print(f"Conclusion: cannot conclude average differs from {reference_mean}.")

In [ ]:
# Simple probability question:
# What's the chance a randomly chosen month (2020–2025 aggregated) exceeds 350 deaths?
threshold = 350
prob_over = np.mean(df["Total Deaths"] > threshold)
print(f"P(Total Deaths > {threshold}) = {prob_over:.2f}")

## 6) Key insights (ready to paste into your report)
- The distribution of monthly totals shows clear **seasonality**. Peaks often appear in late winter/early spring, with relative lows in early fall (your exact results may vary slightly based on your file).
- A few months (e.g., March/June) consistently sit at the **high end** of the range, while months like **October/November** tend to be lower.
- The quick t‑test against a 320 baseline helps **frame the average**—use your p-value to state whether the mean month is statistically different from that reference.
- A simple probability framing (e.g., *P(Total > 350)*) communicates **risk in plain terms** for stakeholders.

## 7) Suggested narrative structure for your short report
1. **Context** – Why overdose seasonality matters (resource planning, outreach timing).  
2. **Data** – 12-month aggregate across 2020–2025; how `Total Deaths` and `Normalized Deaths` were derived.  
3. **Findings** –
   - Distribution (histogram) — spread and center.
   - Seasonal trend (line plot) — which months peak/trough.
   - Top/bottom months (bar) — quick ranking.
   - One-sample t‑test & a simple risk probability.
4. **Implications** – Plan staffing & outreach around high‑risk months; monitor upcoming seasonal peaks.  
5. **Next Steps** – Pull in covariates (weather, policy changes, service capacity); expand to weekly granularity; build an early‑warning baseline model.